In [3]:
import pandas as pd
import json

# Загружаем данные
with open('vacs.json', 'r', encoding='utf-8') as f:
    raw_json = json.load(f)

# Извлекаем содержимое ключа 'data' из каждого блока
processed_data = [item['data'] for item in raw_json if 'data' in item]
df = pd.DataFrame(processed_data)

# Преобразуем даты
df['add_date'] = pd.to_datetime(df['add_date'])

# Расчет годовой зарплаты: (минимум + максимум) / 2 * 12 месяцев
def calculate_annual(row):
    s_min = row.get('salary_min')
    s_max = row.get('salary_max')
    
    values = [v for v in [s_min, s_max] if v is not None]
    if not values:
        return None
    return (sum(values) / len(values)) * 12

df['annual_salary'] = df.apply(calculate_annual, axis=1)

print(f"Всего вакансий в базе: {len(df)}")

Всего вакансий в базе: 1000


In [6]:
# Сортируем по дате добавления
newest_10 = df.sort_values(by='add_date', ascending=False).head(10)

# Выводим название (оно может быть в описании или добавим колонку, если она есть)
newest_10[['add_date', 'salary_formatted']]

,add_date,salary_formatted
367,2023-01-20 16:55:22+03:00,до 80 000 ₽
643,2023-01-20 16:51:27+03:00,47 000 - 52 000 ₽
79,2023-01-20 16:48:14+03:00,до 80 000 ₽
809,2023-01-20 16:43:56+03:00,до 80 000 ₽
629,2023-01-20 16:43:20+03:00,от 70 000 ₽
56,2023-01-20 16:42:09+03:00,до 160 000 ₽
640,2023-01-20 16:36:41+03:00,до 44 626 ₽
896,2023-01-20 16:36:41+03:00,до 44 626 ₽
37,2023-01-20 16:36:41+03:00,до 37 761 ₽
840,2023-01-20 16:36:41+03:00,до 41 193 ₽


In [7]:
# Сортируем по вычисленной годовой зарплате
max_salary_10 = df.sort_values(by='annual_salary', ascending=False).head(10)

max_salary_10[['salary_formatted', 'annual_salary']]

,salary_formatted,annual_salary
418,200 000 - 400 000 ₽,3600000.0
35,40 000 - 550 000 ₽,3540000.0
402,75 000 - 400 000 ₽,2850000.0
541,65 000 - 400 000 ₽,2790000.0
439,153 000 - 236 000 ₽,2334000.0
45,100 000 - 250 000 ₽,2100000.0
145,100 000 - 250 000 ₽,2100000.0
781,114 000 - 228 000 ₽,2052000.0
678,110 000 - 230 000 ₽,2040000.0
239,99 000 - 225 000 ₽,1944000.0


In [8]:
mean_annual = df['annual_salary'].mean()

print(f"Средняя годовая зарплата: {mean_annual:,.2f} ₽")
print(f"Средняя месячная зарплата: {mean_annual / 12:,.2f} ₽")

Средняя годовая зарплата: 913,498.38 ₽
Средняя месячная зарплата: 76,124.87 ₽


In [9]:
# Извлекаем title из словаря working_type
employment_types = df['working_type'].apply(lambda x: x.get('title') if isinstance(x, dict) else None).unique()

print("Типы занятости на рынке:")
for etype in employment_types:
    if etype:
        print(f"- {etype}")

Типы занятости на рынке:
- полная занятость
- частичная занятость
- работа вахтовым методом
- временная работа / freelance
- стажировка


In [10]:
# Фильтруем по названию опыта (обычно 'нет опыта' или 'без опыта')
start_career_df = df[df['experience_length'].apply(
    lambda x: x.get('title') in ['нет опыта', 'без опыта', '0-1 год'] if isinstance(x, dict) else False
)]

count_start = len(start_career_df)
avg_salary_start = start_career_df['annual_salary'].mean()

print(f"Количество вакансий для старта: {count_start}")
if count_start > 0 and not pd.isna(avg_salary_start):
    print(f"Средняя зарплата для новичков: {avg_salary_start / 12:,.2f} ₽/мес.")
else:
    print("Данные о зарплате для старта отсутствуют.")

Количество вакансий для старта: 564
Средняя зарплата для новичков: 75,213.56 ₽/мес.


In [11]:
# В вашем примере есть '3-6 лет'. Если есть 'более 6 лет' — это будет максимумом.
max_exp_val = "более 6 лет" # Попробуйте также "3-6 лет", если "более 6" нет в данных

max_exp_df = df[df['experience_length'].apply(
    lambda x: x.get('title') == max_exp_val if isinstance(x, dict) else False
)]

count_max = len(max_exp_df)
avg_salary_max = max_exp_df['annual_salary'].mean()

print(f"Число вакансий с макс. опытом ({max_exp_val}): {count_max}")
if count_max > 0 and not pd.isna(avg_salary_max):
    print(f"Средняя зарплата опытных специалистов: {avg_salary_max / 12:,.2f} ₽/мес.")

Число вакансий с макс. опытом (более 6 лет): 16
Средняя зарплата опытных специалистов: 138,083.33 ₽/мес.
